# AMPR — Kaggle Run

**Environment:** Kaggle — Python 3.12.12, 2x T4 GPU (15 GB VRAM mỗi card)

Khác với Colab:
- **2x T4**: sử dụng `DataParallel` tự động
- **Persistent storage**: dữ liệu lưu trong Kaggle Dataset
- **Session 12h**: đủ để train cả 3 branches
- **Không mount Drive**: upload `.npy` files vào Kaggle Dataset trước

## Chuẩn bị trước khi chạy notebook này

1. Chạy **Colab notebook** (Phases 2–6) để tạo các `.npy` files
2. Upload `.npy` files lên **Kaggle Dataset**:
   - `seq_embeddings.npy`, `struct_embeddings.npy`, `ppi_embeddings.npy`
   - `labels_mf.npy`, `labels_bp.npy`, `labels_cc.npy`
   - `dag_matrix_mf.npy`, `dag_matrix_bp.npy`, `dag_matrix_cc.npy`
   - `go_emb_mf.npy`, `go_emb_bp.npy`, `go_emb_cc.npy`
   - `splits.json`, `protein_order.json`
3. Attach dataset này vào notebook

In [ ]:
import subprocess, os

# Kiểm tra 2x GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('[GPU CHECK]')
print(result.stdout)

import torch
print(f'torch.cuda.device_count(): {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

---
## Phase 1 — Setup: Clone repo & install

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/ampr.git'
WORK_DIR = '/kaggle/working/ampr'

if not os.path.exists(WORK_DIR):
    !git clone -q {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull -q

os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
!pip install -q torch==2.3.1 transformers==4.41.2
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
!pip install -q obonet==1.0.0 numpy==1.26.4 scikit-learn==1.5.0 pyyaml==6.0.1 tqdm==4.66.4
!pip install -q matplotlib==3.8.4

print('✓ Thư viện đã cài xong!')

---
## Phase 2 — Symlink data từ Kaggle Dataset

Kaggle mount dataset input tại `/kaggle/input/<dataset-name>/`.  
Thay `YOUR_DATASET` bằng tên dataset bạn đã tạo.

In [ ]:
KAGGLE_DATASET = '/kaggle/input/YOUR_DATASET_NAME'  # <-- thay tên dataset

# Kiểm tra files có trong dataset
print('[DATASET FILES]')
!ls -lh {KAGGLE_DATASET}/

# Symlink data/ → Kaggle dataset
DATA_DIR = f'{WORK_DIR}/data'
if not os.path.exists(DATA_DIR):
    os.symlink(KAGGLE_DATASET, DATA_DIR)
    print(f'Symlink: {DATA_DIR} → {KAGGLE_DATASET}')
else:
    print(f'data/ đã tồn tại: {DATA_DIR}')

# Verify key files
required = [
    'data/embeddings/seq_embeddings.npy',
    'data/embeddings/struct_embeddings.npy',
    'data/embeddings/ppi_embeddings.npy',
    'data/labels_mf.npy',
    'data/splits.json',
    'data/protein_order.json',
]
print('\n[VERIFY]')
for f in required:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status}  {f}')

---
## Phase 3 — Cập nhật configs cho Kaggle paths

In [ ]:
import yaml, json

os.makedirs(f'{WORK_DIR}/checkpoints/mf', exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints/bp', exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints/cc', exist_ok=True)
os.makedirs(f'{WORK_DIR}/logs', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

def update_config_paths(config_path, work_dir, suffix='_kaggle'):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    for key in cfg['data']:
        if isinstance(cfg['data'][key], str) and cfg['data'][key].startswith('data/'):
            cfg['data'][key] = os.path.join(work_dir, cfg['data'][key])

    for key in cfg['output']:
        if isinstance(cfg['output'][key], str):
            cfg['output'][key] = os.path.join(work_dir, cfg['output'][key])
            os.makedirs(os.path.dirname(cfg['output'][key]), exist_ok=True)

    out_path = config_path.replace('.yaml', f'{suffix}.yaml')
    with open(out_path, 'w') as f:
        yaml.dump(cfg, f)
    return out_path

mf_config = update_config_paths(f'{WORK_DIR}/configs/mf.yaml', WORK_DIR)
bp_config = update_config_paths(f'{WORK_DIR}/configs/bp.yaml', WORK_DIR)
cc_config = update_config_paths(f'{WORK_DIR}/configs/cc.yaml', WORK_DIR)

print('Kaggle configs generated:')
for c in [mf_config, bp_config, cc_config]:
    print(f'  {c}')

---
## Phase 4 — Train tất cả 3 branches

Kaggle cho phép 12h session, đủ để train 3 models liên tiếp.

DataParallel được kích hoạt tự động khi phát hiện 2 GPU.

In [ ]:
# Train MF
print('=' * 60)
print('TRAINING: Molecular Function (MF) — 2x T4')
print('=' * 60)
!python {WORK_DIR}/main.py --config {mf_config} --seed 42
print('\n[✓] MF done!')

In [ ]:
# Train BP
print('=' * 60)
print('TRAINING: Biological Process (BP) — 2x T4')
print('=' * 60)
!python {WORK_DIR}/main.py --config {bp_config} --seed 42
print('\n[✓] BP done!')

In [ ]:
# Train CC
print('=' * 60)
print('TRAINING: Cellular Component (CC) — 2x T4')
print('=' * 60)
!python {WORK_DIR}/main.py --config {cc_config} --seed 42
print('\n[✓] CC done!')

---
## Phase 5 — Kết quả

Checkpoints được lưu tại `/kaggle/working/ampr/checkpoints/`.  
Kaggle tự động lưu `/kaggle/working/` sau khi session kết thúc — download từ Output tab.

In [ ]:
import re, os

branches = ['mf', 'bp', 'cc']

print('\n' + '='*65)
print(f'{"Branch":>8} | {"Best Fmax":>10} | {"α_seq":>7} | {"α_3di":>7} | {"α_ppi":>7}')
print('-'*65)

epoch_re = re.compile(
    r'EPOCH.*loss=.*bce=.*dag=.*α=\[([\d.]+), ([\d.]+), ([\d.]+)\].*val Fmax=([\d.]+)'
)
fmax_re = re.compile(r'val Fmax=([\d.]+)')

for branch in branches:
    log_path = f'{WORK_DIR}/logs/{branch}_train.log'
    if not os.path.exists(log_path):
        print(f'{branch.upper():>8} | (no log yet)')
        continue

    fmaxes, alphas_seq, alphas_3di, alphas_ppi = [], [], [], []
    with open(log_path) as f:
        for line in f:
            m = epoch_re.search(line)
            if m:
                alphas_seq.append(float(m.group(1)))
                alphas_3di.append(float(m.group(2)))
                alphas_ppi.append(float(m.group(3)))
                fmaxes.append(float(m.group(4)))

    if fmaxes:
        best = fmaxes.index(max(fmaxes))
        print(f'{branch.upper():>8} | {max(fmaxes):>10.3f} | {alphas_seq[best]:>7.3f} | {alphas_3di[best]:>7.3f} | {alphas_ppi[best]:>7.3f}')
    else:
        print(f'{branch.upper():>8} | (parsing failed)')

print('='*65)

print('\n[CHECKPOINTS]')
for branch in branches:
    ckpt = f'{WORK_DIR}/checkpoints/{branch}/best.pt'
    if os.path.exists(ckpt):
        import torch
        data = torch.load(ckpt, map_location='cpu')
        print(f'  {branch.upper()}: epoch={data["epoch"]}, fmax={data["fmax"]:.3f}')
    else:
        print(f'  {branch.upper()}: not found')

---
## Ghi chú Kaggle vs Colab

| Điểm | Colab | Kaggle |
|---|---|---|  
| GPU | 1x T4 15GB | 2x T4 15GB |
| Session | ~4-6h (free) | 12h |
| Storage | Google Drive | Kaggle Dataset (20GB free) |
| DataParallel | Không | Có (auto) |
| Thời gian train 3 models | 2 sessions | 1 session |